# 🏆 Football Prediction App — 02: Feature Engineering

**Amaç:** Geçmiş maç verilerinden model için kullanılabilecek feature'lar üretmek ve 2026 grup fikstürü için tahmin girdileri hazırlamak.

**Kritik kural — Data Leakage yok:** Her feature, o maçın tarihinden **önce** var olan bilgiden hesaplanmalıdır.

---
### Notebook bölümleri
1. Import & veri yükleme
2. Elo rating merge (pre-match)
3. Turnuva ağırlık skoru
4. Rolling form hesaplama
5. Saldırı / savunma gücü
6. Tarihi maç feature matrisi
7. 2026 fikstür feature matrisi
8. Upset risk skoru
9. Kalite kontrolü
10. Çıktıları kaydet

---
## 1. Import & Veri Yükleme

In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', '{:.4f}'.format)

# src klasörünü path'e ekle
SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import FILES, TEAM_NAME_MAP, ROLLING_FORM_WINDOW, BASE_GOAL_RATE

In [ ]:
def load_csv(key, parse_dates=None):
    path = FILES[key]
    if not os.path.isfile(path):
        print(f"[WARN] {key} bulunamadı")
        return pd.DataFrame()
    df = pd.read_csv(path, parse_dates=parse_dates, low_memory=False)
    print(f"[OK]  {key:<18} → {len(df):>7,} satır")
    return df

def standardize(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace(TEAM_NAME_MAP)
    return df

print("Veriler yükleniyor...")
results        = load_csv('results',        parse_dates=['date'])
elo_raw        = load_csv('elo',            parse_dates=['snapshot_date'])
group_fixtures = load_csv('group_fixtures', parse_dates=['date_utc'])
knockout_slots = load_csv('knockout_slots', parse_dates=['date_utc'])

results        = standardize(results,        ['home_team', 'away_team'])
elo_raw        = standardize(elo_raw,        ['country'])
group_fixtures = standardize(group_fixtures, ['home_team', 'away_team'])

print("\nStandardizasyon tamamlandı ✅")

---
## 2. Elo Rating Merge (Pre-Match)

Her maç için o maçın tarihinden **önceki en son** Elo snapshot'ı kullanılır.  
Böylece gelecek bilgisi sızması (data leakage) önlenir.

In [ ]:
# Elo tablosundan her (ülke, tarih) için bir rating satırı oluştur
elo = elo_raw[['snapshot_date', 'country', 'rating', 'rank', 'confederation']].copy()
elo = elo.sort_values('snapshot_date')

def get_elo_at_date(team: str, date: pd.Timestamp) -> float:
    """
    Verilen tarihten önceki (veya eşit) en son Elo ratingini döner.
    Bulunamazsa NaN döner.
    """
    sub = elo[(elo['country'] == team) & (elo['snapshot_date'] <= date)]
    if sub.empty:
        return np.nan
    return sub.iloc[-1]['rating']

# Hızlı lookup için pivot: her takımın tüm snapshot'larını dict olarak tut
elo_lookup = {}
for country, grp in elo.groupby('country'):
    elo_lookup[country] = grp[['snapshot_date', 'rating']].set_index('snapshot_date')['rating']

def fast_elo(team: str, date: pd.Timestamp) -> float:
    if team not in elo_lookup:
        return np.nan
    series = elo_lookup[team]
    past = series[series.index <= date]
    return float(past.iloc[-1]) if not past.empty else np.nan

# Test
test_teams = ['Brazil', 'France', 'Argentina', 'Germany']
test_date  = pd.Timestamp('2022-11-20')
print(f"{'Takım':<15} {'Elo @ ' + str(test_date.date()):>20}")
print('-' * 36)
for t in test_teams:
    print(f"{t:<15} {fast_elo(t, test_date):>20.1f}")

---
## 3. Turnuva Ağırlık Skoru

Dünya Kupası maçı bir dostluk maçından çok daha önemli.  
Ağırlık sistemi feature'larda ve rolling form hesabında kullanılır.

In [ ]:
TOURNAMENT_WEIGHTS = {
    # Maç tipi         : ağırlık
    'FIFA World Cup'         : 4.0,
    'UEFA Euro'              : 3.5,
    'Copa America'           : 3.5,
    'Africa Cup of Nations'  : 3.0,
    'AFC Asian Cup'          : 3.0,
    'CONCACAF Gold Cup'      : 3.0,
    'UEFA Nations League'    : 2.5,
    'Confederations Cup'     : 2.5,
    'World Cup qualification': 2.0,
    'UEFA Euro qualification': 1.5,
    'Friendly'               : 1.0,
}

def get_tournament_weight(tournament: str) -> float:
    if pd.isna(tournament):
        return 1.0
    for key, w in TOURNAMENT_WEIGHTS.items():
        if key.lower() in tournament.lower():
            return w
    return 1.5  # Tanınmayan turnuva — dostluktan biraz önemli

# Turnuva dağılımını göster
if not results.empty:
    results['tournament_weight'] = results['tournament'].apply(get_tournament_weight)
    print("Turnuva ağırlık dağılımı (top 15):")
    tw = (
        results.groupby('tournament')['tournament_weight']
        .first()
        .sort_values(ascending=False)
        .head(15)
    )
    display(tw.to_frame('ağırlık'))

---
## 4. Rolling Form Hesaplama

Her takım için maç tarihinden önceki son N maçtaki:
- `points_last_n` — kazanma=3, beraberlik=1, mağlubiyet=0
- `goal_diff_last_n` — atılan − yenilen
- `gf_last_n` — atılan goller
- `ga_last_n` — yenilen goller
- `weighted_form` — son maçlara daha yüksek ağırlık

In [ ]:
def get_team_matches(results_df: pd.DataFrame, team: str) -> pd.DataFrame:
    """Bir takımın ev sahibi veya deplasman olduğu tüm maçları döner."""
    home = results_df[results_df['home_team'] == team].copy()
    home['is_home'] = True
    home['goals_for']     = home['home_score']
    home['goals_against'] = home['away_score']

    away = results_df[results_df['away_team'] == team].copy()
    away['is_home'] = False
    away['goals_for']     = away['away_score']
    away['goals_against'] = away['home_score']

    combined = pd.concat([home, away], ignore_index=True)
    combined = combined.dropna(subset=['goals_for', 'goals_against'])
    combined['goals_for']     = combined['goals_for'].astype(int)
    combined['goals_against'] = combined['goals_against'].astype(int)

    def get_points(row):
        if row['goals_for'] > row['goals_against']:  return 3
        if row['goals_for'] == row['goals_against']: return 1
        return 0

    combined['points'] = combined.apply(get_points, axis=1)
    combined = combined.sort_values('date').reset_index(drop=True)
    return combined


def get_rolling_form(results_df: pd.DataFrame, team: str,
                     current_date: pd.Timestamp, n: int = ROLLING_FORM_WINDOW) -> dict:
    """Verilen tarihten önceki son n maçın form istatistiklerini döner."""
    team_matches = get_team_matches(results_df, team)
    past = team_matches[team_matches['date'] < current_date].tail(n)

    if past.empty:
        return {
            'points_last_n'   : np.nan,
            'goal_diff_last_n': np.nan,
            'gf_last_n'       : np.nan,
            'ga_last_n'       : np.nan,
            'weighted_form'   : np.nan,
            'matches_played'  : 0,
        }

    # Son maçlara üstel ağırlık: en son maç en yüksek ağırlığa sahip
    weights = np.exp(np.linspace(0, 1, len(past)))  # [0.37, ..., 1.0] arası
    weights /= weights.sum()

    weighted_pts = (past['points'].values * weights).sum() * n  # n ile normalize

    return {
        'points_last_n'   : past['points'].sum(),
        'goal_diff_last_n': (past['goals_for'] - past['goals_against']).sum(),
        'gf_last_n'       : past['goals_for'].sum(),
        'ga_last_n'       : past['goals_against'].sum(),
        'weighted_form'   : weighted_pts,
        'matches_played'  : len(past),
    }

# Test
test_result = get_rolling_form(results, 'Brazil', pd.Timestamp('2022-11-20'))
print("Brazil form @ 2022-11-20 (son 5 maç):")
for k, v in test_result.items():
    print(f"  {k:<22}: {v}")

---
## 5. Saldırı / Savunma Gücü

Son `window` maçtaki gol ortalamaları baz alınarak hesaplanır.  
**Attack strength** = takımın gol atma kapasitesi / global ortalama  
**Defense weakness** = takıma atılan gol ortalaması / global ortalama

In [ ]:
def get_attack_defense(results_df: pd.DataFrame, team: str,
                       current_date: pd.Timestamp, window: int = 20) -> dict:
    """Takımın saldırı gücü ve savunma zayıflığını döner."""
    team_matches = get_team_matches(results_df, team)
    past = team_matches[team_matches['date'] < current_date].tail(window)

    if past.empty or len(past) < 3:
        return {'attack_strength': 1.0, 'defense_weakness': 1.0, 'form_matches': 0}

    # Global ortalama: tüm tarihi verideki gol/maç
    global_avg = BASE_GOAL_RATE  # config'den gelir (1.35)

    avg_gf = past['goals_for'].mean()
    avg_ga = past['goals_against'].mean()

    attack_strength  = avg_gf / global_avg if global_avg > 0 else 1.0
    defense_weakness = avg_ga / global_avg if global_avg > 0 else 1.0

    return {
        'attack_strength' : round(attack_strength, 4),
        'defense_weakness': round(defense_weakness, 4),
        'form_matches'    : len(past),
    }

# Test — top takımlar
test_date = pd.Timestamp('2026-06-01')
teams_test = ['Brazil', 'France', 'Argentina', 'Germany', 'Morocco', 'South Korea']
rows = []
for t in teams_test:
    ad = get_attack_defense(results, t, test_date)
    rows.append({'takım': t, **ad})
display(pd.DataFrame(rows).set_index('takım').round(3))

---
## 6. Tarihi Maç Feature Matrisi

Model eğitimi için her geçmiş maçı bir satıra dönüştürüyoruz.  
Her satır: `[elo_diff, form_diff, attack_home, defense_home, attack_away, defense_away, neutral, tournament_weight, target]`

> **Not:** Bu adım büyük veri seti için birkaç dakika sürebilir (~45k maç).

In [ ]:
# Performans için: sadece 2000 sonrası maçları kullan (gürültüyü azaltır)
results_modern = results[results['date'] >= '2000-01-01'].copy()
results_modern = results_modern.dropna(subset=['home_score', 'away_score'])
results_modern = results_modern.reset_index(drop=True)

print(f"2000 sonrası maç sayısı: {len(results_modern):,}")
print("Feature matrisi oluşturuluyor (bu işlem birkaç dakika sürebilir)...")

In [ ]:
from tqdm.auto import tqdm

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

feature_rows = []
iterable = tqdm(results_modern.iterrows(), total=len(results_modern)) if HAS_TQDM else results_modern.iterrows()

for idx, row in iterable:
    home = row['home_team']
    away = row['away_team']
    date = row['date']

    # --- Elo ---
    elo_home = fast_elo(home, date)
    elo_away = fast_elo(away, date)
    elo_diff = elo_home - elo_away if (not np.isnan(elo_home) and not np.isnan(elo_away)) else np.nan

    # --- Rolling Form ---
    form_home = get_rolling_form(results, home, date)
    form_away = get_rolling_form(results, away, date)
    form_diff = (
        form_home['weighted_form'] - form_away['weighted_form']
        if (not np.isnan(form_home['weighted_form']) and not np.isnan(form_away['weighted_form']))
        else np.nan
    )

    # --- Attack / Defense ---
    ad_home = get_attack_defense(results, home, date)
    ad_away = get_attack_defense(results, away, date)

    # --- Target ---
    hs, as_ = int(row['home_score']), int(row['away_score'])
    if hs > as_:   result_label = 'H'
    elif hs == as_: result_label = 'D'
    else:           result_label = 'A'

    feature_rows.append({
        # Kimlik
        'date'             : date,
        'home_team'        : home,
        'away_team'        : away,
        # Elo
        'elo_home'         : elo_home,
        'elo_away'         : elo_away,
        'elo_diff'         : elo_diff,          # home - away
        # Form
        'form_home_pts'    : form_home['points_last_n'],
        'form_away_pts'    : form_away['points_last_n'],
        'form_diff'        : form_diff,          # weighted form farkı
        'weighted_form_home': form_home['weighted_form'],
        'weighted_form_away': form_away['weighted_form'],
        # Gol
        'gf_home_last_n'   : form_home['gf_last_n'],
        'ga_home_last_n'   : form_home['ga_last_n'],
        'gf_away_last_n'   : form_away['gf_last_n'],
        'ga_away_last_n'   : form_away['ga_last_n'],
        # Attack / Defense
        'attack_home'      : ad_home['attack_strength'],
        'defense_home'     : ad_home['defense_weakness'],
        'attack_away'      : ad_away['attack_strength'],
        'defense_away'     : ad_away['defense_weakness'],
        # Bağlam
        'neutral'          : int(row.get('neutral', 0)),
        'tournament_weight': get_tournament_weight(row.get('tournament', '')),
        # Hedef
        'home_score'       : hs,
        'away_score'       : as_,
        'result'           : result_label,       # H / D / A
    })

features_df = pd.DataFrame(feature_rows)
print(f"\nFeature matrisi hazır: {features_df.shape}")
display(features_df.head(3))

In [ ]:
# Hedef dağılımı
print("Hedef (H/D/A) dağılımı:")
vc = features_df['result'].value_counts()
for label, count in vc.items():
    bar = '█' * int(count / len(features_df) * 40)
    print(f"  {label}: {count:>6,}  ({count/len(features_df)*100:.1f}%)  {bar}")

# Eksik değer raporu
print("\nEksik değer oranları (feature kolonları):")
feat_cols = ['elo_diff', 'form_diff', 'attack_home', 'defense_home', 'attack_away', 'defense_away']
for c in feat_cols:
    pct = features_df[c].isna().mean() * 100
    print(f"  {c:<25}: {pct:.1f}%")

In [ ]:
# Feature korelasyon görselleştirmesi
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, feat, xlabel in zip(
    axes,
    ['elo_diff', 'form_diff', 'attack_home'],
    ['Elo Farkı (Home - Away)', 'Weighted Form Farkı', 'Home Saldırı Gücü']
):
    for label, color in [('H','steelblue'), ('D','gray'), ('A','coral')]:
        subset = features_df[features_df['result'] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.5, color=color, label=label, density=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Yoğunluk')
    ax.legend(title='Sonuç')
    ax.set_title(f'{xlabel}\n→ Sonuç Ayrımı')

plt.tight_layout()
plt.show()

---
## 7. 2026 Fikstür Feature Matrisi

Gelecek maçlar için feature hesaplama — aynı fonksiyonlar, tarih = maç tarihi.

In [ ]:
future_rows = []

for _, row in group_fixtures.iterrows():
    home = row['home_team']
    away = row['away_team']
    date = row['date_utc']

    elo_home = fast_elo(home, date)
    elo_away = fast_elo(away, date)
    elo_diff = elo_home - elo_away if (not np.isnan(elo_home) and not np.isnan(elo_away)) else np.nan

    form_home = get_rolling_form(results, home, date)
    form_away = get_rolling_form(results, away, date)
    form_diff = (
        form_home['weighted_form'] - form_away['weighted_form']
        if (not np.isnan(form_home['weighted_form']) and not np.isnan(form_away['weighted_form']))
        else np.nan
    )

    ad_home = get_attack_defense(results, home, date)
    ad_away = get_attack_defense(results, away, date)

    future_rows.append({
        'match_id'         : row['match_id'],
        'group'            : row['group'],
        'date_utc'         : date,
        'venue'            : row['venue'],
        'home_team'        : home,
        'away_team'        : away,
        'elo_home'         : elo_home,
        'elo_away'         : elo_away,
        'elo_diff'         : elo_diff,
        'form_home_pts'    : form_home['points_last_n'],
        'form_away_pts'    : form_away['points_last_n'],
        'form_diff'        : form_diff,
        'weighted_form_home': form_home['weighted_form'],
        'weighted_form_away': form_away['weighted_form'],
        'gf_home_last_n'   : form_home['gf_last_n'],
        'ga_home_last_n'   : form_home['ga_last_n'],
        'gf_away_last_n'   : form_away['gf_last_n'],
        'ga_away_last_n'   : form_away['ga_last_n'],
        'attack_home'      : ad_home['attack_strength'],
        'defense_home'     : ad_home['defense_weakness'],
        'attack_away'      : ad_away['attack_strength'],
        'defense_away'     : ad_away['defense_weakness'],
        'neutral'          : 1,   # Dünya Kupası maçları nötr sahada
        'tournament_weight': 4.0, # FIFA World Cup
    })

future_df = pd.DataFrame(future_rows)
print(f"2026 fikstür feature matrisi: {future_df.shape}")
print()

# Elo'su eksik takımlar
missing_elo = future_df[future_df['elo_diff'].isna()][['home_team','away_team','elo_home','elo_away']]
if not missing_elo.empty:
    print(f"⚠️  Elo eksik olan maçlar ({len(missing_elo)}):")
    display(missing_elo)
else:
    print("✅  Tüm maçlar için Elo mevcut")

print("\nİlk 5 maç:")
display(future_df[['match_id','group','home_team','away_team','elo_home','elo_away','elo_diff','form_diff']].head())

In [ ]:
# Gruba göre Elo sıralaması
print("GRUP BAZINDA ELO SIRALAMALARI")
print("=" * 50)

# Her takımın son Elo'sunu al
ref_date = pd.Timestamp('2026-06-01')
all_teams = pd.concat([
    future_df[['group', 'home_team']].rename(columns={'home_team': 'team'}),
    future_df[['group', 'away_team']].rename(columns={'away_team': 'team'})
]).drop_duplicates()

all_teams['elo'] = all_teams['team'].apply(lambda t: fast_elo(t, ref_date))
all_teams = all_teams.sort_values(['group', 'elo'], ascending=[True, False])

for grp in sorted(all_teams['group'].unique()):
    sub = all_teams[all_teams['group'] == grp]
    print(f"\nGrup {grp}:")
    for _, r in sub.iterrows():
        elo_val = f"{r['elo']:.0f}" if not np.isnan(r['elo']) else 'N/A'
        print(f"  {r['team']:<30} Elo: {elo_val}")

---
## 8. Upset Risk Skoru

Favorinin kırılganlığını ölçen açıklayıcı metrik.  
**Model çıktısı değil** — dashboard'da kullanıcıya gösterilen yardımcı skor.

In [ ]:
def compute_upset_risk(elo_diff: float, underdog_form: float,
                       favorite_form: float, neutral: int) -> float:
    """
    Upset risk = 0-1 arası skor (1 = en yüksek sürpriz riski).
    Formül döküman §7.2'den alınmıştır.
    """
    if np.isnan(elo_diff) or elo_diff == 0:
        return 0.5  # Fark yoksa orta risk

    abs_diff = abs(elo_diff)

    # 1. Elo farkı ne kadar küçükse risk o kadar yüksek
    max_elo_diff = 600.0
    elo_component = 1 - min(abs_diff / max_elo_diff, 1.0)

    # 2. Underdog formunun gücü
    underdog_form_score = 0.5  # varsayılan
    if not np.isnan(underdog_form) and underdog_form > 0:
        underdog_form_score = min(underdog_form / (ROLLING_FORM_WINDOW * 3), 1.0)

    # 3. Favori negatif momentumu
    fav_neg_momentum = 0.5  # varsayılan
    if not np.isnan(favorite_form) and favorite_form > 0:
        # Favori düşük form puanı alıyorsa momentum zayıf
        fav_neg_momentum = 1 - min(favorite_form / (ROLLING_FORM_WINDOW * 3), 1.0)

    # 4. Nötr saha bağlamı
    neutral_component = 0.3 if neutral else 0.0

    # Ağırlıklı toplam
    upset_risk = (
        0.40 * elo_component
        + 0.25 * underdog_form_score
        + 0.20 * fav_neg_momentum
        + 0.10 * neutral_component
        + 0.05 * 0.5  # high_goal_variance_context — sabit tahmini
    )
    return round(min(max(upset_risk, 0.0), 1.0), 4)


def upset_label(score: float) -> str:
    if score >= 0.65: return '🔴 Yüksek'
    if score >= 0.45: return '🟡 Orta'
    return '🟢 Düşük'


# 2026 fikstürüne uygula
future_df['upset_risk'] = future_df.apply(
    lambda r: compute_upset_risk(
        r['elo_diff'],
        r['weighted_form_away'],   # underdog = away (elo_diff > 0 ise)
        r['weighted_form_home'],   # favori   = home
        r['neutral']
    ),
    axis=1
)
future_df['upset_label'] = future_df['upset_risk'].apply(upset_label)

print("2026 Maçları — Upset Risk Sıralaması (yüksekten düşüğe):")
display(
    future_df[['group','home_team','away_team','elo_diff','upset_risk','upset_label']]
    .sort_values('upset_risk', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

---
## 9. Kalite Kontrolü

In [ ]:
print("TARİHİ FEATURE MATRİSİ — ÖZET")
print("=" * 50)
print(f"Toplam maç       : {len(features_df):,}")
print(f"Tarih aralığı    : {features_df['date'].min().date()}  →  {features_df['date'].max().date()}")
print()

key_features = ['elo_diff', 'form_diff', 'attack_home', 'defense_home',
                'attack_away', 'defense_away', 'neutral', 'tournament_weight']

summary = features_df[key_features].describe().T[['mean', 'std', 'min', 'max']]
display(summary.round(3))

print("\n2026 FİKSTÜR FEATURE MATRİSİ — ÖZET")
print("=" * 50)
print(f"Toplam maç       : {len(future_df)}")
elo_coverage = future_df['elo_diff'].notna().mean() * 100
print(f"Elo coverage     : {elo_coverage:.1f}%")
form_coverage = future_df['form_diff'].notna().mean() * 100
print(f"Form coverage    : {form_coverage:.1f}%")

---
## 10. Çıktıları Kaydet

In [ ]:
PROCESSED_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

features_path = os.path.join(PROCESSED_DIR, 'features_historical.csv')
future_path   = os.path.join(PROCESSED_DIR, 'features_2026_fixtures.csv')

features_df.to_csv(features_path, index=False)
future_df.to_csv(future_path, index=False)

print(f"✅  Tarihi feature matrisi kaydedildi   → {features_path}")
print(f"✅  2026 fikstür feature matrisi kaydedildi → {future_path}")
print(f"\nDosya boyutları:")
print(f"  features_historical.csv    : {os.path.getsize(features_path)/1024:.1f} KB")
print(f"  features_2026_fixtures.csv : {os.path.getsize(future_path)/1024:.1f} KB")

---
## 11. Sonraki Adım

> **`03_model_training.ipynb`** → 
> - `features_historical.csv` yükle
> - Time-based train/valid/test split
> - Logistic Regression baseline (H/D/A)
> - Poisson expected-goals modeli
> - Log loss, Brier score, kalibrasyon değerlendirmesi
> - Model kaydet (`joblib`)
> - 2026 fikstürü için tahmin üret